In [ ]:
!pip install ultralytics opencv-python-headless numpy -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.6 MB/s eta 0:00:00


In [ ]:
# Rebuild Model

import zipfile, os, shutil, torch

MODEL_DIR = "/content/drive/MyDrive/SafeCommuteAI/NOT/vehicle.pt"
OUTPUT_PT = "/content/vehicle_fixed.pt"
TEMP_ZIP  = "/content/vehicle_temp.zip"

for root, dirs, files in os.walk(MODEL_DIR):
    for f in files:
        os.utime(os.path.join(root, f), (1609459200, 1609459200))

with zipfile.ZipFile(TEMP_ZIP, "w", compression=zipfile.ZIP_STORED) as zf:
    for root, dirs, files in os.walk(MODEL_DIR):
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, MODEL_DIR)
            zf.write(full_path, arcname)

shutil.copy(TEMP_ZIP, OUTPUT_PT)
ckpt = torch.load(OUTPUT_PT, map_location="cpu", weights_only=False)
print("Model OK!!! Keys:", list(ckpt.keys()))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Model OK!!! Keys: ['date', 'version', 'license', 'docs', 'epoch', 'best_fitness', 'model', 'ema', 'updates', 'optimizer', 'scaler', 'train_args', 'train_metrics', 'train_results', 'git']


In [ ]:
# Copy Images to Local (faster than reading from Drive)

import shutil, os

SRC = "/content/drive/MyDrive/SafeCommuteAI/NOT"
DST = "/content/images"
OUT = "/content/cropped"

os.makedirs(DST, exist_ok=True)
os.makedirs(OUT, exist_ok=True)

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff", ".tif"}

copied = 0
for f in os.listdir(SRC):
    if os.path.splitext(f)[1].lower() in IMAGE_EXTS:
        shutil.copy(os.path.join(SRC, f), os.path.join(DST, f))
        copied += 1

print(f"Copied {copied} images to {DST}")

Copied 1377 images to /content/images


In [ ]:
# Run Inference (GPU + Fixed Square Crop)

import os, cv2, numpy as np
from ultralytics import YOLO

INPUT_DIR  = "/content/images"
OUTPUT_DIR = "/content/cropped"
MODEL_PATH = "/content/vehicle_fixed.pt"

BUS_CLASS_ID   = 0
CONF_THRESHOLD = 0.4
PADDING_RATIO  = 0.20
BATCH_SIZE     = 16
IMAGE_EXTS     = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff", ".tif"}

def make_square_crop(img, x1, y1, x2, y2):
    H, W = img.shape[:2]
    bw, bh = x2 - x1, y2 - y1

    pad_x = int(bw * PADDING_RATIO)
    pad_y = int(bh * PADDING_RATIO)
    x1p, y1p = x1 - pad_x, y1 - pad_y
    x2p, y2p = x2 + pad_x, y2 + pad_y

    cx = (x1p + x2p) // 2
    cy = (y1p + y2p) // 2
    side = max(x2p - x1p, y2p - y1p)
    half = side // 2

    x1s, y1s = cx - half, cy - half
    x2s, y2s = x1s + side, y1s + side

    # Shift to stay inside — no padding, no mirroring
    if x1s < 0: x2s -= x1s; x1s = 0
    if y1s < 0: y2s -= y1s; y1s = 0
    if x2s > W: x1s -= (x2s - W); x2s = W
    if y2s > H: y1s -= (y2s - H); y2s = H

    # Hard clamp
    x1s, y1s = max(0, x1s), max(0, y1s)
    x2s, y2s = min(W, x2s), min(H, y2s)

    crop = img[y1s:y2s, x1s:x2s]

    # If slightly non-square (bus at image edge), resize cleanly — NO border padding
    ch, cw = crop.shape[:2]
    if ch != cw:
        target = max(ch, cw)
        crop = cv2.resize(crop, (target, target), interpolation=cv2.INTER_LANCZOS4)

    return crop

print("[INFO] Loading model on GPU...")
model = YOLO(MODEL_PATH)

image_files = sorted([
    os.path.join(INPUT_DIR, f)
    for f in os.listdir(INPUT_DIR)
    if os.path.splitext(f)[1].lower() in IMAGE_EXTS
])
print(f"[INFO] Found {len(image_files)} images\n")

total_crops = 0
no_detect   = 0
skipped     = 0

for batch_start in range(0, len(image_files), BATCH_SIZE):
    batch_paths = image_files[batch_start : batch_start + BATCH_SIZE]

    imgs, valid_paths = [], []
    for p in batch_paths:
        img = cv2.imread(p)
        if img is None:
            print(f"  SKIPPED: {os.path.basename(p)}")
            skipped += 1
            continue
        imgs.append(img)
        valid_paths.append(p)

    if not imgs:
        continue

    results = model.predict(
        source=valid_paths,
        conf=CONF_THRESHOLD,
        classes=[BUS_CLASS_ID],
        device=0,
        verbose=False,
        stream=True,
    )

    for img, img_path, result in zip(imgs, valid_paths, results):
        basename = os.path.splitext(os.path.basename(img_path))[0]
        boxes = result.boxes

        if boxes is None or len(boxes) == 0:
            print(f"[>] {os.path.basename(img_path)} ... no buses detected")
            no_detect += 1
            continue

        bus_count = 0
        for i, box in enumerate(boxes):
            if int(box.cls[0].item()) != BUS_CLASS_ID:
                continue
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            crop = make_square_crop(img, x1, y1, x2, y2)
            if crop.size == 0:
                continue
            cv2.imwrite(
                os.path.join(OUTPUT_DIR, f"{basename}_bus_{i}.jpg"),
                crop, [cv2.IMWRITE_JPEG_QUALITY, 95]
            )
            bus_count += 1
            total_crops += 1

        print(f"[>] {os.path.basename(img_path)} ... {bus_count} crop(s)")

print(f"\n{'='*50}")
print(f" Total crops  : {total_crops}")
print(f"   No detection : {no_detect}")
print(f"   Skipped      : {skipped}")
print(f"{'='*50}")

[INFO] Loading model on GPU...
[INFO] Found 1377 images

[>] 01000.jpg ... 1 crop(s)
[>] 01001.jpg ... 1 crop(s)
[>] 01002.jpg ... 1 crop(s)
[>] 01003.jpg ... 1 crop(s)
[>] 01004.jpg ... 1 crop(s)
[>] 01005.jpg ... 2 crop(s)
[>] 01006.jpg ... 1 crop(s)
[>] 01007.jpg ... 2 crop(s)
[>] 01008.jpg ... 2 crop(s)
[>] 01009.jpg ... 1 crop(s)
[>] 01010.jpg ... 1 crop(s)
[>] 01011.jpg ... 1 crop(s)
[>] 01012.jpg ... 1 crop(s)
[>] 01013.jpg ... 3 crop(s)
[>] 01014.jpg ... 1 crop(s)
[>] 01015.jpg ... 1 crop(s)
[>] 01016.jpg ... 1 crop(s)
[>] 01017.jpg ... 1 crop(s)
[>] 01018.jpg ... 1 crop(s)
[>] 01019.jpg ... 1 crop(s)
[>] 01020.jpg ... 2 crop(s)
[>] 01021.jpg ... 1 crop(s)
[>] 01022.jpg ... 1 crop(s)
[>] 01023.jpg ... 1 crop(s)
[>] 01024.jpg ... 1 crop(s)
[>] 01025.jpg ... 1 crop(s)
[>] 01026.jpg ... 1 crop(s)
[>] 01027.jpg ... 1 crop(s)
[>] 01028.jpg ... 5 crop(s)
[>] 01029.jpg ... 1 crop(s)
[>] 01030.jpg ... 1 crop(s)
[>] 01031.jpg ... 1 crop(s)
[>] 01032.jpg ... 1 crop(s)
[>] 01033.jpg ... 1

In [ ]:
# Verifying Square Cropping

import cv2, os, random

files = [f for f in os.listdir("/content/cropped") if f.endswith(".jpg")]
sample = random.sample(files, min(10, len(files)))

print("Checking 10 random crops:")
all_ok = True
for f in sample:
    img = cv2.imread(f"/content/cropped/{f}")
    h, w = img.shape[:2]
    ok = "✅" if h == w else "❌"
    print(f"  {ok} {f}: {w}x{h}")
    if h != w: all_ok = False

print(f"\n{'All square!!!' if all_ok else '***Some not square, check logic***!!!'}")

Checking 10 random crops:
  ✅ 0939_bus_0.jpg: 1577x1577
  ✅ 01423_bus_0.jpg: 1423x1423
  ✅ 01929_bus_1.jpg: 856x856
  ✅ 02176_bus_0.jpg: 574x574
  ✅ 0842_bus_1.jpg: 224x224
  ✅ 01105_bus_1.jpg: 1304x1304
  ✅ 01947_bus_0.jpg: 1008x1008
  ✅ 01341_bus_0.jpg: 1591x1591
  ✅ 0830_bus_0.jpg: 563x563
  ✅ 01569_bus_0.jpg: 375x375

All square!!!


In [ ]:
# Copy to Drive + Download

import shutil
from google.colab import files

# Copy cropped folder back to Drive
DRIVE_OUT = "/content/drive/MyDrive/SafeCommuteAI/NOT/cropped_fixed"
if os.path.exists(DRIVE_OUT):
    shutil.rmtree(DRIVE_OUT)
shutil.copytree("/content/cropped", DRIVE_OUT)
print(f"Saved to Drive: {DRIVE_OUT}")

# Also zip + download locally
shutil.make_archive("/content/cropped_fixed", "zip", "/content/cropped")
files.download("/content/cropped_fixed.zip")

Saved to Drive: /content/drive/MyDrive/SafeCommuteAI/NOT/cropped_fixed


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>